# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, following the Croissant schema standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading

Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview

Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List all record sets and their fields using their @id
record_sets = dataset.record_sets  # List of CroissantRecordSet objects
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for i, rs in enumerate(record_sets):
        print(f"[{i}] Record Set: {rs.name} (@id: {rs.id})")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}) | type: {field.data_type if hasattr(field, 'data_type') else 'N/A'}")
        print("")
    # For demo: choose the first record set
    selected_record_set_id = record_sets[0].id if record_sets else None


## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Reference all entities by their `@id`.

In [ ]:
# Generate a mapping from record set @id to its name
dataframes = {}

# Use .id for the Croissant @id
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded record set '{rs.name}' (@id: {rs.id}) with shape {df.shape}")
    else:
        print(f"No records found for '{rs.name}' (@id: {rs.id})\n")

if not dataframes:
    raise ValueError("No data frames extracted from record sets!")

# Pick one record set (the first) to display columns and head
main_record_set_id = list(dataframes.keys())[0]
main_df = dataframes[main_record_set_id]

print(f"\nColumns available in record set (@id: {main_record_set_id}):")
print(list(main_df.columns))
main_df.head()

## 4. Exploratory Data Analysis (EDA)

We will apply some basic exploratory steps: filtering records, normalizing a selected numeric field, and grouping records.

**Note:** All fields are referenced by their `@id`. Please review the printed columns above to pick suitable fields for your analysis.

In [ ]:
# Select a numeric field for demonstration, e.g. 'Age_at_Second_CRC', referencing its @id
# Replace below with the exact @id from your printed main_df columns if available
# For illustration, suppose the @id is 'Age_at_Second_CRC'
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    # Look for likely numeric (e.g. age, year, interval, etc.)
    if ('age' in col.lower() or 'age' in col) and numeric_field_id is None:
        numeric_field_id = col
    # A categorical/grouping field (e.g. 'Sex', 'MSI Status', 'Anatomical Location')
    if ('sex' in col.lower() or 'status' in col.lower() or 'location' in col.lower()) and group_field_id is None:
        group_field_id = col
        
if not numeric_field_id:
    numeric_field_id = main_df.select_dtypes(include=['int', 'float']).columns[0]  # first numeric col
    print(f"Auto-selected numeric field: {numeric_field_id}")
if not group_field_id:
    # fallback to another likely field
    group_field_id = main_df.columns[1] if main_df.shape[1]>1 else main_df.columns[0]
    print(f"Auto-selected categorical field: {group_field_id}")

threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
filtered = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records where '{numeric_field_id}' > {threshold} : {filtered.shape[0]} rows")

# Normalization
if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    filtered = filtered.copy()
    filtered[numeric_field_id + '_normalized'] = (
        (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
    )
    print(f"\nExample normalization for field '{numeric_field_id}':")
    print(filtered[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Grouping
if group_field_id in filtered.columns and pd.api.types.is_numeric_dtype(filtered[numeric_field_id]):
    grouped = filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped average '{numeric_field_id}' by '{group_field_id}':")
    print(grouped.head())

## 5. Visualization

Visualize distributions or relationships in the data, using the selected fields and referencing by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid", palette="muted", font_scale=1.1)

# Histogram of the numeric field (referenced by @id)
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group_field_id (if suitable)
if group_field_id and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    plt.figure(figsize=(10,5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated the use of the `mlcroissant` library to load, inspect, and analyze a clinical research dataset package defined by a Croissant schema. All fields and record sets were referenced by their `@id` as per best practices. 

- **Dataset loaded and metadata reviewed.**
- **All record sets and fields explored by `@id`.**
- **Extracted data, filtered, normalized, and grouped using fields identified by `@id`.**
- **Visualized key distributions in the data.**

You can adapt this notebook to focus deeper on specific fields or statistical analyses according to research needs. For more, consult the [Croissant documentation](https://mlcommons.github.io/croissant/).
